# Testing endpoints

In [19]:
import json
import email_tools

# List all emails
content = email_tools.list_all_emails()
print(json.dumps(content, indent=2))

[
  {
    "id": 5,
    "sender": "you@email.com",
    "recipient": "boss@email.com",
    "subject": "Re: Quarterly Report",
    "body": "Dear Boss,\n\nI wanted to follow up regarding your recent email about the quarterly report. I have received your message and will prioritize the work as requested. Please let me know if there is anything specific you would like to add or change.\n\nBest regards,\n",
    "timestamp": "2026-04-22T13:53:20.273963",
    "read": false
  },
  {
    "id": 1,
    "sender": "boss@email.com",
    "recipient": "you@email.com",
    "subject": "Quarterly Report",
    "body": "Please finalize the report ASAP.",
    "timestamp": "2026-04-22T13:34:53.279276",
    "read": true
  },
  {
    "id": 2,
    "sender": "alice@work.com",
    "recipient": "you@email.com",
    "subject": "Lunch?",
    "body": "Free for lunch today?",
    "timestamp": "2026-04-22T13:34:53.279276",
    "read": false
  },
  {
    "id": 3,
    "sender": "eric@work.com",
    "recipient": "you@email.

In [20]:
# Send a new email
new_email = email_tools.send_email("test@example.com", "Lunch plans", "Shall we meet at noon?")
print(json.dumps(new_email, indent=2))

{
  "id": 6,
  "sender": "you@email.com",
  "recipient": "test@example.com",
  "subject": "Lunch plans",
  "body": "Shall we meet at noon?",
  "timestamp": "2026-04-22T13:56:06.656120",
  "read": false
}


In [3]:
# Get email by id
email_tools.get_email(4)

{'id': 4,
 'sender': 'bob@work.com',
 'recipient': 'you@email.com',
 'subject': 'Code Review',
 'body': 'I left some comments on your PR.',
 'timestamp': '2026-04-22T13:34:53.279276',
 'read': False}

In [4]:
# Search emails by keyword
email_tools.search_emails("lunch")

[{'id': 5,
  'sender': 'you@email.com',
  'recipient': 'test@example.com',
  'subject': 'Lunch plans',
  'body': 'Shall we meet at noon?',
  'timestamp': '2026-04-22T13:34:55.031529',
  'read': False},
 {'id': 2,
  'sender': 'alice@work.com',
  'recipient': 'you@email.com',
  'subject': 'Lunch?',
  'body': 'Free for lunch today?',
  'timestamp': '2026-04-22T13:34:53.279276',
  'read': False}]

In [5]:
# filter emails 
email_tools.filter_emails(recipient="test@example.com")

[{'id': 5,
  'sender': 'you@email.com',
  'recipient': 'test@example.com',
  'subject': 'Lunch plans',
  'body': 'Shall we meet at noon?',
  'timestamp': '2026-04-22T13:34:55.031529',
  'read': False}]

In [6]:
# unread emails
email_tools.list_unread_emails()

[{'id': 5,
  'sender': 'you@email.com',
  'recipient': 'test@example.com',
  'subject': 'Lunch plans',
  'body': 'Shall we meet at noon?',
  'timestamp': '2026-04-22T13:34:55.031529',
  'read': False},
 {'id': 1,
  'sender': 'boss@email.com',
  'recipient': 'you@email.com',
  'subject': 'Quarterly Report',
  'body': 'Please finalize the report ASAP.',
  'timestamp': '2026-04-22T13:34:53.279276',
  'read': False},
 {'id': 2,
  'sender': 'alice@work.com',
  'recipient': 'you@email.com',
  'subject': 'Lunch?',
  'body': 'Free for lunch today?',
  'timestamp': '2026-04-22T13:34:53.279276',
  'read': False},
 {'id': 3,
  'sender': 'eric@work.com',
  'recipient': 'you@email.com',
  'subject': 'Happy Hour',
  'body': "We're planning drinks this Friday!",
  'timestamp': '2026-04-22T13:34:53.279276',
  'read': False},
 {'id': 4,
  'sender': 'bob@work.com',
  'recipient': 'you@email.com',
  'subject': 'Code Review',
  'body': 'I left some comments on your PR.',
  'timestamp': '2026-04-22T13:34:5

In [7]:
# mark as read
email_tools.mark_email_as_read(new_email['id'])

{'id': 5,
 'sender': 'you@email.com',
 'recipient': 'test@example.com',
 'subject': 'Lunch plans',
 'body': 'Shall we meet at noon?',
 'timestamp': '2026-04-22T13:34:55.031529',
 'read': True}

In [8]:
# mark as unread
email_tools.mark_email_as_unread(new_email['id'])

{'id': 5,
 'sender': 'you@email.com',
 'recipient': 'test@example.com',
 'subject': 'Lunch plans',
 'body': 'Shall we meet at noon?',
 'timestamp': '2026-04-22T13:34:55.031529',
 'read': False}

In [9]:
# delete email
email_tools.delete_email(new_email['id'])

{'message': 'Email deleted'}

# Email assistant agent

In [13]:
import os
from dotenv import load_dotenv
import aisuite as ai

load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
client = ai.Client()

In [14]:
def email_assistant(request: str):
    prompt = f"""
    - You are an AI assistant specialized in managing emails.
    - You can perform various actions such as listing, searching, filtering, and manipulating emails.
    - Use the provided tools to interact with the email system.
    - Never ask the user for confirmation before performing an action.
    - If needed, my email address is "you@email.com" so you can use it to send emails or perform actions related to my account.
    {request.strip()}
    """
    
    response = client.chat.completions.create(
        model="openai:gpt-4.1",
        messages=[
            {
                "role":"user",
                "content":(prompt)
            }
        ],
        tools = [
            email_tools.search_unread_from_sender,
            email_tools.list_unread_emails,
            email_tools.search_emails,
            email_tools.get_email,
            email_tools.mark_email_as_read,
            email_tools.send_email,
            email_tools.delete_email
        ],
        max_turns=5,
    )
    
    return response

In [18]:
res = email_assistant("Check for unread emails from boss@email.com, mark them as read, and send a polite follow-up.")
res

ChatCompletion(id='chatcmpl-DXSKZkDJt1UZuKO2uT912Zl23ECGL', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='There are no unread emails from boss@email.com in your inbox. No emails need to be marked as read, and no follow-up message will be sent at this time. Let me know if you need help with other email actions!', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), intermediate_messages=[ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_N68spaou2P7gJBgIAROGNmMl', function=Function(arguments='{"sender_email":"boss@email.com"}', name='search_unread_from_sender'), type='function')]), {'role': 'tool', 'name': 'search_unread_from_sender', 'content': '[]', 'tool_call_id': 'call_N68spaou2P7gJBgIAROGNmMl'}, ChatCompletionMessage(content='There are no unread emails from boss

In [21]:
res = email_assistant("Delete alice@work.com email.")
res

ChatCompletion(id='chatcmpl-DXSNki5QxM9PGFUfImFBeGOsBl5ag', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The email from alice@work.com has been deleted. If you need to manage any other emails or need further assistance, let me know!', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), intermediate_messages=[ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_ct2m2fmzVhQVTi2yoYXefmTV', function=Function(arguments='{"sender_email":"alice@work.com"}', name='search_unread_from_sender'), type='function')]), {'role': 'tool', 'name': 'search_unread_from_sender', 'content': '[{"id": 2, "sender": "alice@work.com", "recipient": "you@email.com", "subject": "Lunch?", "body": "Free for lunch today?", "timestamp": "2026-04-22T13:34:53.279276", "read": false}]', 'tool_call_id': 

In [22]:
res = email_assistant("Delete the happy hour email")
res

ChatCompletion(id='chatcmpl-DXSOh5DJVdE7dYfjUbRueUqNocJzR', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The happy hour email has been deleted. If you need any further actions with your emails, let me know!', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), intermediate_messages=[ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_NPILflTq6hATxAwoqwHSRgh2', function=Function(arguments='{"query":"happy hour"}', name='search_emails'), type='function')]), {'role': 'tool', 'name': 'search_emails', 'content': '[{"id": 3, "sender": "eric@work.com", "recipient": "you@email.com", "subject": "Happy Hour", "body": "We\'re planning drinks this Friday!", "timestamp": "2026-04-22T13:34:53.279276", "read": false}]', 'tool_call_id': 'call_NPILflTq6hATxAwoqwHSRgh2'}, ChatComple